# Khoi tao trong so va Vanishing/Exploding Gradient voi tf.keras

**Muc tieu bai hoc:**
- Hieu vi sao **cach khoi tao trong so ban dau** anh huong quyet dinh den viec mang no-ron co hoc
  duoc hay khong - du kien truc va du lieu giu nguyen.
- Hieu 2 hien tuong **vanishing gradient** (gradient tien dan ve 0, cac lop dau hoc rat cham hoac
  khong hoc) va **exploding gradient** (gradient/gia tri tang vot, cost dao dong hoac thanh `NaN`).
- Biet dung `kernel_initializer` cua `tf.keras.layers.Dense` de chon chien luoc khoi tao phu hop,
  va hieu vi sao **He initialization** thuong la lua chon mac dinh tot cho mang dung ReLU.

In [ ]:
# --- Google Colab setup: bo qua tu dong neu chay Jupyter local ---
import sys, os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # tat toi uu oneDNN de dam bao ket qua giong het nhau
# giua cac lan chay - phai dat TRUOC khi 'import tensorflow' o cell sau (oneDNN co the doi thu
# tu tinh toan tren nhieu luong CPU, gay lech ket qua nho o vai chu so thap phan cuoi giua cac lan)

if "google.colab" in sys.modules:
    REPO_DIR = "/content/AI-Teaching-Labs"
    if not os.path.isdir(REPO_DIR):
        !git clone --depth 1 https://github.com/NonomiyaIzumi/AI-Teaching-Labs.git {REPO_DIR}
    NOTEBOOK_DIR = f"{REPO_DIR}/Deep Learning/Module 1/keras-tensorflow/03_Vanishing-Exploding-gradient-va-Khoi-tao-trong-so"
    os.chdir(NOTEBOOK_DIR)
    print("Da chuyen working directory sang:", os.getcwd())
else:
    print("Khong phai Colab - dung working directory hien tai:", os.getcwd())
    print("Neu chay Jupyter local: tu thu muc keras-tensorflow/, chay 'uv sync' truoc.")

## 1. Vi sao khoi tao trong so lai quan trong?

Truoc khi huan luyen, moi trong so `W^[l]` can duoc gan mot gia tri ban dau. Ba cach khoi tao pho
bien - va he qua cua tung cach - la:

### 1.1. Khoi tao bang 0 (`zeros`)

Neu **moi** trong so trong cung mot lop bat dau bang 0, moi neuron trong lop do se luon nhan duoc
CUNG mot gradient trong suot qua trinh huan luyen (vi dau vao giong nhau, trong so giong nhau ->
dau ra giong nhau -> dao ham giong nhau). Cac neuron "dong bo" voi nhau mai mai - day goi la
**van de doi xung (symmetry)**: du co bao nhieu neuron trong mot lop, chung hoc y het nhau, tuong
duong nhu lop do chi co **dung 1 neuron**. Mang khong bao gio hoc duoc bieu dien phong phu.

### 1.2. Khoi tao ngau nhien nhung QUA LON (vd `randn(...) * 10`)

Pha vo duoc tinh doi xung (moi trong so khac nhau ngau nhien), nhung neu do lon (phuong sai) qua
cao, gia tri `Z^[l] = W^{[l]}A^{[l-1]}+b^{[l]}` o cac lop sau se rat lon ve tri tuyet doi. Voi
Sigmoid/Tanh, `|Z|` lon dua ham vao vung **bao hoa** (dao ham gan bang 0) - gradient lan truyen
nguoc qua nhieu lop se **nhan don dan lai** (vanishing gradient), cac lop dau hoc rat cham. Voi
ReLU, `Z` qua lon/qua am co the day mang vao trang thai bat on dinh, cost co the dao dong manh hoac
"no" thanh gia tri rat lon (`exploding gradient`).

### 1.3. Khoi tao co kiem soat theo do sau mang (**Xavier/Glorot**, **He**)

Y tuong chung: chon **phuong sai** cua trong so ti le nghich voi so neuron dau vao cua lop (`fan_in`)
de **giu on dinh phuong sai cua `Z^[l]` qua cac lop** - khong qua nho (vanishing) cung khong qua
lon (exploding):

$$\text{Xavier/Glorot: } W^{[l]} \sim \mathcal{N}\!\left(0, \frac{1}{n^{[l-1]}}\right)
\qquad\qquad
\text{He: } W^{[l]} \sim \mathcal{N}\!\left(0, \frac{2}{n^{[l-1]}}\right)$$

**He initialization** (He et al., 2015) nhan them he so 2 - phu hop hon cho **ReLU**, vi ReLU "cat"
mot nua truc so (moi gia tri am ve 0), nen can phuong sai lon hon o dau vao de bu lai phan bi mat
do sau khi qua ReLU. Day la ly do He thuong la lua chon mac dinh cho cac mang dung ReLU (truong
hop pho bien nhat trong thuc te).

## 2. Khoi tao trong so trong `tf.keras`

Moi lop `layers.Dense(...)` co tham so `kernel_initializer` (cho trong so `W`) va `bias_initializer`
(cho `b`, mac dinh la 0 - hop ly vi bias khong gay ra van de doi xung nhu `W`). Ba chien luoc o muc
1 deu co san duoi dang class trong `tf.keras.initializers`:

| Chien luoc | Class Keras |
|---|---|
| Khoi tao bang 0 | `keras.initializers.Zeros()` |
| Ngau nhien, phuong sai lon (`stddev=10`) | `keras.initializers.RandomNormal(stddev=10.0)` |
| He initialization | `keras.initializers.HeNormal()` |
| Xavier/Glorot initialization | `keras.initializers.GlorotNormal()` (**mac dinh cua `Dense`** neu khong chi dinh gi) |

## 3. Du lieu & kien truc thi nghiem

De quan sat ro hien tuong vanishing/exploding, ta dung mot mang **du sau** (`[8, 10, 5, 1]`, 2
hidden layer) tren bo du lieu y te that **Pima Indians Diabetes** (768 benh nhan, 8 dac trung lam
sang, du doan tieu duong).

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers

from keras_utils import load_dataset, compile_and_train, evaluate_classification

tf.config.experimental.enable_op_determinism()  # dam bao ket qua giong het nhau giua cac lan chay

train_X, train_Y, test_X, test_Y = load_dataset()
print("train_X:", train_X.shape, " train_Y:", train_Y.shape)
print("test_X: ", test_X.shape, " test_Y: ", test_Y.shape)

## 4. Xay dung mang voi initializer tuy chinh - Bai tap

### Bai tap - `build_model_with_initializer`

Ham nhan vao mot `initializer` bat ky va ap dung **cung mot** initializer do cho ca 3 lop `Dense`,
de co the goi lai voi `Zeros()`, `RandomNormal(stddev=10.0)`, hay `HeNormal()` va so sanh cong bang.

In [ ]:
def build_model_with_initializer(input_dim, initializer):
    '''
    Xay dung kien truc [input_dim, 10, 5, 1] (Dense -> ReLU -> Dense -> ReLU -> Dense -> Sigmoid),
    ap dung CUNG MOT initializer cho ca 3 lop Dense.

    Arguments:
    input_dim   -- so dac trung dau vao (8 cho Pima Diabetes)
    initializer -- mot tf.keras.initializers.Initializer (Zeros / RandomNormal / HeNormal ...)

    Returns: mot tf.keras.Model CHUA compile.
    '''
    # (~6 dong) dung keras.Sequential + layers.Dense(..., kernel_initializer=initializer)
    # model = keras.Sequential([
    #     keras.Input(shape=(input_dim,)),
    #     layers.Dense(10, activation="relu", kernel_initializer=initializer),
    #     layers.Dense(5, activation="relu", kernel_initializer=initializer),
    #     layers.Dense(1, activation="sigmoid", kernel_initializer=initializer),
    # ])
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE
    return model

In [ ]:
#@title Answer { display-mode: "form" }
def build_model_with_initializer(input_dim, initializer):
    '''
    Xay dung kien truc [input_dim, 10, 5, 1] (Dense -> ReLU -> Dense -> ReLU -> Dense -> Sigmoid),
    ap dung CUNG MOT initializer cho ca 3 lop Dense.

    Arguments:
    input_dim   -- so dac trung dau vao (8 cho Pima Diabetes)
    initializer -- mot tf.keras.initializers.Initializer (Zeros / RandomNormal / HeNormal ...)

    Returns: mot tf.keras.Model CHUA compile.
    '''
    # (~6 dong) dung keras.Sequential + layers.Dense(..., kernel_initializer=initializer)
    # YOUR CODE STARTS HERE
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(10, activation="relu", kernel_initializer=initializer),
        layers.Dense(5, activation="relu", kernel_initializer=initializer),
        layers.Dense(1, activation="sigmoid", kernel_initializer=initializer),
    ])
    # YOUR CODE ENDS HERE
    return model

## 5. Huan luyen & so sanh 3 cach khoi tao

Ca 3 lan train dung chung `learning_rate=0.01`, full-batch gradient descent, chi khac o
`kernel_initializer` - de moi chenh lech ket qua chi den tu cach khoi tao, khong den tu yeu to nao
khac.

In [ ]:
initializers = {
    "zeros":      keras.initializers.Zeros(),
    "random_bad": keras.initializers.RandomNormal(mean=0.0, stddev=10.0, seed=3),
    "he":         keras.initializers.HeNormal(seed=3),
}

EPOCHS = 2000
histories = {}
for name, init in initializers.items():
    tf.random.set_seed(3)
    model = build_model_with_initializer(train_X.shape[0], init)
    history = compile_and_train(
        model, train_X, train_Y,
        optimizer=keras.optimizers.SGD(learning_rate=0.01),
        epochs=EPOCHS, batch_size=None, verbose=0,
    )
    histories[name] = {"model": model, "history": history}
    final_cost = history.history["loss"][-1]
    print(f"{name:10s} -> cost cuoi cung (train) = {final_cost:.4f}")

In [ ]:
plt.figure(figsize=(7, 4.5))
for name in initializers:
    plt.plot(histories[name]["history"].history["loss"], label=name)
plt.xlabel("epoch")
plt.ylabel("cost")
plt.title("Cost qua qua trinh train - so sanh kernel_initializer")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for name in initializers:
    evaluate_classification(histories[name]["model"], test_X, test_Y, f'kernel_initializer = "{name}" (test set)')

## 6. Ket qua & Phan tich

Sau 2000 epoch (`learning_rate=0.01`), ket qua tren test set: **`zeros`** 64.9% accuracy nhung
**Precision = Recall = 0** - mo hinh khong phat hien duoc benh nhan nao mac benh, dung du doan tren
duoc mo ta o muc 1.1 (van de doi xung khien mang khong hoc duoc gi khac ngoai mot du doan "phang").
**`random_bad`** (`stddev=10`) cung 64.9% accuracy, Recall = 0 - cost co giam nhe (con 0.5413) nhung
van khong tach duoc lop nao, dung nhu du bao o muc 1.2 (logit qua lon lam bao hoa activation).
**`he`** dat **68.2%** accuracy, Recall **51.9%** - hoc duoc mot tin hieu du doan that su. Ca 3 con
so nay se con tot hon neu tang so epoch (o day chi dung 2000 epoch de notebook chay nhanh, van du
de thay ro su khac biet dinh tinh giua 3 cach khoi tao).

**Diem mau chot:** cung mot kien truc, cung mot du lieu, cung mot thuat toan toi uu - chi **mot**
tham so khac nhau (cach khoi tao trong so ban dau) da tao ra chenh lech tu "khong hoc duoc gi" den
"hoc duoc mot mo hinh co ich". Day la ly do khoi tao trong so la mot trong nhung quyet dinh dau
tien va quan trong nhat khi thiet ke mang no-ron - va cung la ly do cac framework hien dai (Keras,
PyTorch) deu chon san **He/Xavier lam mac dinh** cho cac lop Dense/Conv, thay vi de nguoi dung tu
chon ngau nhien.

## 7. Bai tap mo rong

1. **Xavier/Glorot cho ReLU:** thu doi `he` thanh `keras.initializers.GlorotNormal(seed=3)` - ket
   qua co kem hon `HeNormal` khong? Giai thich bang cong thuc o muc 1.3 (thieu he so 2 cho ReLU).
2. **Mang sau hon:** tang kien truc thanh `[8, 20, 20, 20, 10, 5, 1]` (5 hidden layer) - `zeros` va
   `random_bad` co "hong" nang hon khong khi mang sau hon? `he` co con on dinh khong?
3. **Theo doi gradient flow:** dung `tf.GradientTape` de tinh va ghi lai `\|\|dW^{[l]}\|\|` (chuan
   Frobenius cua gradient tung lop) sau moi vai epoch cho ca 3 cach khoi tao, roi ve bieu do -
   quan sat truc tiep gradient "vanish" (tien ve 0) hay on dinh qua cac lop nhu the nao.
4. **Batch/Layer Normalization:** so sanh cach tiep can "chon khoi tao tot" (bai nay) voi cach tiep
   can "chuan hoa lai `Z` sau moi lop trong luc train" (BatchNormalization/LayerNormalization) -
   ca hai co the giai quyet cung mot van de vanishing/exploding gradient theo hai huong khac nhau.

## 8. Tai lieu tham khao

| Chu de | Lien ket |
|---|---|
| He et al., 2015 - Delving Deep into Rectifiers (He initialization) | https://arxiv.org/abs/1502.01852 |
| Glorot & Bengio, 2010 - Understanding the difficulty of training deep feedforward neural networks | https://proceedings.mlr.press/v9/glorot10a.html |
| `tf.keras.initializers` API | https://www.tensorflow.org/api_docs/python/tf/keras/initializers |
| Pima Indians Diabetes Dataset (nguon goc) | https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database |